# Выполнение запросов при загрузке данных с PostGIS
## Загружаем данные из PostGIS в Spark (temp view) и далее выполняем запросы на Sedona

# Инициализация

In [2]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .master("local[*]") \
    .getOrCreate()

sedona = SedonaContext.create(config)


# Декоратор для замера скорости выполнения запроса

In [3]:
import time
from functools import wraps

def timer_sedona(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

In [4]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

# Загрузка данных

In [5]:
# @timer_sedona
def download_data(session, query):
    result_df = session.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({query}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
    result_df.createOrReplaceTempView("spatial_table")

# Выполнение запроса

In [6]:
@timer_sedona
def execute_query(session, download_query, query):
    # new_df = loaded_df.selectExpr(query)
    download_data(session, download_query)
    result_df = session.sql(query)
    result_df.show(10)

In [7]:
def execute_query_all(session, download_query, query):
    download_query_1 = download_query + "1"
    print("1 запись: ")
    execute_query(session, download_query_1, query)
    
    download_query_10 = download_query + "10"
    print("\n 10 записей: ")
    execute_query(session, download_query_10, query)
    
    download_query_100 = download_query + "100"
    print("\n 100 записей: ")
    execute_query(session, download_query_100, query)
    
    download_query_1k = download_query + "1000"
    print("\n 1K записей: ")
    execute_query(session, download_query_1k, query)
    
    download_query_10k = download_query + "10000"
    print("\n 10K записей: ")
    execute_query(session, download_query_10k, query)
    
    download_query_100k = download_query + "100000"
    print("\n 100K записей: ")
    execute_query(session, download_query_100k, query)

# Вычисление площади

In [11]:
# запрос
# sql_area = """SELECT id, layer_id, ST_Area(ST_Transform(ST_SetSRID(geometry, 4326), 'EPSG:3857')) as area FROM public.features_plain LIMIT """
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """SELECT id, layer_id, ST_Area(ST_Transform(ST_SetSRID(ST_GeomFromWKB(geometry), 4326), 'EPSG:3857')) as area FROM spatial_table"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+------------------+
|     id|layer_id|              area|
+-------+--------+------------------+
|3734070|     170|10451.548012536385|
+-------+--------+------------------+

Функция execute_query выполнена за 0.5749 секунд

 10 записей: 
+-------+--------+------------------+
|     id|layer_id|              area|
+-------+--------+------------------+
|3734070|     170|10451.548012536385|
|3734071|     170|3102.7913124126676|
|3734072|     170|13369.078504725354|
|3734073|     170|11077.198155459306|
|3734074|     170|4035.9218524714474|
|3734088|     170|2806.7097767579967|
|3734075|     170|33163.726682253175|
|3734076|     170|3366.9368842756153|
|3734077|     170|10841.050560861904|
|3746406|     172| 9944.652194705845|
+-------+--------+------------------+

Функция execute_query выполнена за 0.6001 секунд

 100 записей: 
+-------+--------+------------------+
|     id|layer_id|              area|
+-------+--------+------------------+
|3734070|     170|1045

# Вычисление геометрия полигона в WKT формате

In [10]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """SELECT id, layer_id, ST_AsText(ST_GeomFromWKB(geometry)) as geom_wkt FROM spatial_table"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|            geom_wkt|
+-------+--------+--------------------+
|3674762|     170|MULTIPOLYGON (((3...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.6480 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|            geom_wkt|
+-------+--------+--------------------+
|3674762|     170|MULTIPOLYGON (((3...|
|3738867|     170|MULTIPOLYGON (((3...|
|3674766|     170|MULTIPOLYGON (((3...|
|3674770|     170|MULTIPOLYGON (((3...|
|3677928|     170|MULTIPOLYGON (((3...|
|3674767|     170|MULTIPOLYGON (((3...|
|3674768|     170|MULTIPOLYGON (((3...|
|3674769|     170|MULTIPOLYGON (((3...|
|3576219|     168|POINT (37.5388215...|
|3674771|     170|MULTIPOLYGON (((3...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.7234 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|            geom_wkt|
+-------+--------+

# Центроид полигона в WKT формате

In [15]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """SELECT id, layer_id, ST_AsText(ST_Centroid(ST_GeomFromWKB(geometry))) as centroid_wkt FROM spatial_table"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|        centroid_wkt|
+-------+--------+--------------------+
|3634563|     169|POINT (37.5277277...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5514 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|        centroid_wkt|
+-------+--------+--------------------+
|3634563|     169|POINT (37.5277277...|
|3634564|     169|POINT (37.656838 ...|
|3634565|     169|POINT (37.4544125...|
|3634566|     169|POINT (37.5348459...|
|3634567|     169|POINT (37.4559549...|
|3634568|     169|POINT (37.758545 ...|
|3634569|     169|POINT (37.5968542...|
|3634570|     169|POINT (37.7612474...|
|3634571|     169|POINT (37.5697048...|
|3634572|     169|POINT (37.4637241...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5265 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|        centroid_wkt|
+-------+--------+

# Значение периметра полигона

In [17]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """SELECT id, layer_id, ST_Perimeter(ST_Transform(ST_SetSRID(ST_GeomFromWKB(geometry), 4326), 'EPSG:3857'))
 as perimeter FROM spatial_table"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+-----------------+
|     id|layer_id|        perimeter|
+-------+--------+-----------------+
|3669975|     170|1856.038958587164|
+-------+--------+-----------------+

Функция execute_query выполнена за 0.6029 секунд

 10 записей: 
+-------+--------+------------------+
|     id|layer_id|         perimeter|
+-------+--------+------------------+
|3669975|     170| 1856.038958587164|
|3672173|     170|1481.3031801355114|
|3576973|     168|               0.0|
|3669976|     170|1663.6564899374582|
|3669977|     170| 1117.965422891608|
|3576923|     168|               0.0|
|3669978|     170|1690.8280295549973|
|3669979|     170|264.67429506288886|
|3669980|     170|477.56221562712784|
|3576512|     168|               0.0|
+-------+--------+------------------+

Функция execute_query выполнена за 0.6081 секунд

 100 записей: 
+-------+--------+------------------+
|     id|layer_id|         perimeter|
+-------+--------+------------------+
|3669975|     170| 1856.038

# Координаты центроида полигона

In [19]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """
WITH calc_centroid AS (
    SELECT 
        id,
        layer_id,
        ST_Centroid(ST_GeomFromWKB(geometry)) AS centroid 
    FROM spatial_table
)
SELECT 
    id,
    layer_id,
    ST_X(centroid) AS X_centroid,
    ST_Y(centroid) AS Y_centroid
FROM calc_centroid
"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+-----------------+-----------------+
|     id|layer_id|       X_centroid|       Y_centroid|
+-------+--------+-----------------+-----------------+
|3730250|     170|37.12167307068899|55.51935602232264|
+-------+--------+-----------------+-----------------+

Функция execute_query выполнена за 0.6504 секунд

 10 записей: 
+-------+--------+------------------+------------------+
|     id|layer_id|        X_centroid|        Y_centroid|
+-------+--------+------------------+------------------+
|3730250|     170| 37.12167307068899| 55.51935602232264|
|3730255|     170|   37.617691080299| 55.71393061246113|
|3730251|     170| 37.30037173402106|55.479175507997844|
|3730252|     170| 37.41547685295618| 55.66704418983882|
|3745777|     172| 37.80180400285741| 55.72401910271763|
|3730253|     170|37.414123910382024| 55.66814707699724|
|3730254|     170| 37.41225186522752| 55.66975594814411|
|3730256|     170|37.539610329142604| 55.66219486478321|
|3730257|     170| 37.

# Переставленные координаты центроида полигона

In [21]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """
    with flipped_cenroid_cord as (
        SELECT id, layer_id, ST_FlipCoordinates(ST_Centroid(ST_GeomFromWKB(geometry))) as centroid FROM spatial_table )
        select id, layer_id, ST_X(centroid) as X_centroid, ST_Y(centroid) as Y_centroid FROM flipped_cenroid_cord"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+------------+------------+
|     id|layer_id|  X_centroid|  Y_centroid|
+-------+--------+------------+------------+
|3423269|     164|55.598214935|37.198180243|
+-------+--------+------------+------------+

Функция execute_query выполнена за 0.5258 секунд

 10 записей: 
+-------+--------+------------+------------+
|     id|layer_id|  X_centroid|  Y_centroid|
+-------+--------+------------+------------+
|3423269|     164|55.598214935|37.198180243|
|3423272|     164|   55.509676|   37.284414|
|3423273|     164|55.889434511|37.582740915|
|3423274|     164|55.739793954|37.547647529|
|3423275|     164|   55.734339|    37.47518|
|3423276|     164|   55.725422|   37.813989|
|3423277|     164|55.765933503|37.743233446|
|3423278|     164| 55.73405759| 37.62280253|
|3423279|     164|55.813002398|37.784953687|
|3423280|     164| 55.84267208| 37.42744742|
+-------+--------+------------+------------+

Функция execute_query выполнена за 0.5432 секунд

 100 записей: 
+--

# Центроид полигона в WKT формате с переставленными координатами

In [22]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """
    with flipped_cenroid_cord as (
        select id, layer_id, ST_FlipCoordinates(ST_Centroid(ST_GeomFromWKB(geometry))) as centroid FROM spatial_table )
        select id, layer_id, ST_AsText(centroid) as flipped_centroid FROM flipped_cenroid_cord"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+--------------------+
|3630116|     169|POINT (55.7952289...|
+-------+--------+--------------------+

Функция execute_query выполнена за 1.2814 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+--------------------+
|3630116|     169|POINT (55.7952289...|
|3630117|     169|POINT (55.8030831...|
|3630118|     169|POINT (55.8562968...|
|3630119|     169|POINT (55.8052069...|
|3630120|     169|POINT (55.7411419...|
|3630121|     169|POINT (55.7235098...|
|3630122|     169|POINT (55.6765118...|
|3630123|     169|POINT (55.7328325...|
|3630124|     169|POINT (55.7197321...|
|3630125|     169|POINT (55.7099902...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5223 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+

# Геометрия полигона с переставленными координатами в WKT формате

In [23]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """ SELECT id, layer_id, ST_FlipCoordinates(ST_GeomFromWKB(geometry)) as flipped_geoemtry FROM spatial_table """
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+--------------------+
|3549234|     166|POLYGON ((55.8179...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5707 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+--------------------+
|3549234|     166|POLYGON ((55.8179...|
|3665514|     170|MULTIPOLYGON (((5...|
|3665515|     170|MULTIPOLYGON (((5...|
|3665516|     170|MULTIPOLYGON (((5...|
|3665517|     170|MULTIPOLYGON (((5...|
|3666074|     170|MULTIPOLYGON (((5...|
|3549235|     166|POLYGON ((55.7368...|
|3665522|     170|MULTIPOLYGON (((5...|
|3665523|     170|MULTIPOLYGON (((5...|
|3665524|     170|MULTIPOLYGON (((5...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5702 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+

# Ближайшие объекты. 
## Находим ближайшие объекты к данному с помощью KNN

In [8]:
df = sedona.read \
        .format("jdbc") \
        .option("url", postgresql_url) \
        .option("user", credentials.get('user')) \
        .option("password", credentials.get('password')) \
        .option("dbtable", "public.features_plain") \
        .option("driver", "org.postgresql.Driver") \
        .load()
    
# 2. Регистрируем как временную таблицу
df.createOrReplaceTempView("features_plain")

In [27]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """

sql_area = """
WITH source_geometry AS (
    SELECT ST_GeomFromWKB(geometry) as geometry
    FROM spatial_table
    LIMIT 1
)
SELECT 
    st.id,
    st.layer_id,
    ST_Distance(so.geometry, ST_GeomFromWKB(st.geometry)) as distance
FROM source_geometry so
CROSS JOIN spatial_table st
WHERE ST_Distance(so.geometry, ST_GeomFromWKB(st.geometry)) IS NOT NULL
ORDER BY distance
LIMIT 50
"""
execute_query_all(sedona, sql_download_data, sql_area)


1 запись: 
+-------+--------+--------+
|     id|layer_id|distance|
+-------+--------+--------+
|3510305|     165|     0.0|
+-------+--------+--------+

Функция execute_query выполнена за 0.9507 секунд

 10 записей: 
+-------+--------+-------------------+
|     id|layer_id|           distance|
+-------+--------+-------------------+
|3510305|     165|                0.0|
|3510306|     165|0.10486160776992842|
|3510307|     165|0.10486269364148111|
|3510311|     165|0.11863125629348435|
|3510310|     165|0.12372830248549214|
|3738833|     170|0.12785776251783132|
|3663561|     170|0.16047652972620846|
|3510324|     165|0.16503490930571207|
|3510309|     165| 0.4898143888851502|
|3510308|     165|0.48981658663742417|
+-------+--------+-------------------+

Функция execute_query выполнена за 0.9943 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|            distance|
+-------+--------+--------------------+
|3664650|     170|0.009819354542663737|
|3510569|   

# Упрощение геометрии объекта: ST_SimplifyPreserveTopology

In [29]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """ SELECT id, layer_id, ST_SimplifyPreserveTopology(ST_GeomFromWKB(geometry), 10) as simplified_geoemtry FROM spatial_table LIMIT """
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3379696|     163|POINT (37.8178420...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5232 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3379696|     163|POINT (37.8178420...|
|3414688|     164|POINT (37.8155465...|
|3414689|     164|POINT (37.4855190...|
|3414690|     164|POINT (37.714637 ...|
|3414691|     164|POINT (37.7143765...|
|3414692|     164|POINT (37.5124900...|
|3414693|     164|POINT (37.5906375...|
|3414694|     164|POINT (37.6882068...|
|3414695|     164|POINT (37.733842 ...|
|3414696|     164|POINT (37.7148106...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.4705 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+

# Определить административную принадлежность объекта к округу и району
## Забираем актуальные границы оркгуов и районов из базы

In [30]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/mkgh_monitorings"

In [31]:
sql_get_regions = """select * from nsi.nsi_moscow_regions"""
df_regions = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_regions}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_regions.show(10)

+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|region_id|           full_name|                name|short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
|  1| 11001200|Троицкий и Новомо...|Троицки

In [32]:
sql_get_districts = """select * from nsi.nsi_moscow_districts"""
df_districts = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_districts}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_districts.show(10)
df_districts.createOrReplaceTempView("districts_table")

+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|district_id|           full_name|          name|        short_name|region_id|   region_name|region_short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------

In [33]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """

sql_query = """
WITH source_objects AS (
    SELECT 
        id,
        layer_id,
        ST_Transform(ST_MakeValid(ST_SetSRID(ST_GeomFromWKB(geometry), 4326)), 'EPSG:3857') AS geometry_3857 
    FROM spatial_table
),
districts_transformed as (
    select  district_id, short_name, region_id, region_short_name,  ST_SetSRID(ST_GeomFromWKB(geometry_3857), 3857) as geometry_3857
    from districts_table
    ),
intersections_with_districts AS (
    SELECT 
        so.id,
        so.layer_id,
        md.district_id,
        md.short_name AS short_district_name,
        md.region_id,
        md.region_short_name AS region_short_name,
        ST_Area(ST_Intersection(so.geometry_3857, md.geometry_3857)) AS area_intersection
    FROM districts_transformed AS md
    CROSS JOIN source_objects AS so
    WHERE ST_Intersects(so.geometry_3857, md.geometry_3857)
),
ranked_intersections AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY id, layer_id ORDER BY area_intersection DESC) AS rn
    FROM intersections_with_districts
)
SELECT 
    id,
    layer_id,
    district_id,
    short_district_name,
    region_id,
    region_short_name,
    area_intersection
FROM ranked_intersections
WHERE rn = 1
"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3622300|     169|POINT (36.9414665...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5223 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3622300|     169|POINT (36.9414665...|
|3622301|     169|POINT (37.1872578...|
|3622302|     169|POINT (37.5240918...|
|3622303|     169|POINT (37.524309 ...|
|3622304|     169|POINT (37.3497864...|
|3622305|     169|POINT (37.7191694...|
|3622306|     169|POINT (37.5988063...|
|3622307|     169|POINT (37.4789366...|
|3622308|     169|POINT (36.9365724...|
|3622309|     169|POINT (37.535219 ...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5803 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+